In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("choongqianzheng/disease-and-symptoms-dataset")

print("Path to dataset files:", path)

c:\Users\visha\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 24.1k/24.1k [00:00<00:00, 224kB/s]

Extracting files...
Path to dataset files: C:\Users\visha\.cache\kagglehub\datasets\choongqianzheng\disease-and-symptoms-dataset\versions\2


In [6]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 1. Update with your actual local file path
file_path = "C:/Users/visha/.cache/kagglehub/datasets/choongqianzheng/disease-and-symptoms-dataset/versions/2/Disease precaution.csv" 

# 2. Load and Clean
df = pd.read_csv(file_path)

# This dataset often has empty spaces in strings, let's clean them
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# 3. Identify Target and Features
# In this dataset, 'Disease' is the column we want to predict
X = df.drop('Disease', axis=1)
y = df['Disease']

# Convert symptoms (text) into dummy variables (numbers)
X_encoded = pd.get_dummies(X)

# 4. Train
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# 5. Save the Model and the Column names (Important for prediction later!)
joblib.dump(model, 'disease_model.pkl')
joblib.dump(X_encoded.columns, 'model_columns.pkl')

print(f"✅ Model trained on {len(df)} records and saved!")
print(f"Accuracy: {model.score(X_test, y_test)*100:.2f}%")

✅ Model trained on 41 records and saved!
Accuracy: 0.00%


In [7]:
import joblib
import pandas as pd
import numpy as np

# 1. Load the model and the expected column structure
model = joblib.load('disease_model.pkl')
model_columns = joblib.load('model_columns.pkl')

# 2. Create a blank input with all zeros
# This ensures we have the same 100+ symptom columns the model expects
input_data = pd.DataFrame(0, index=[0], columns=model_columns)

# 3. SET YOUR SYMPTOMS HERE (Change these to 1 to 'activate' them)
# Note: The column names usually look like 'Symptom_1_itching'
# You can print(model_columns) to see all available symptoms
symptoms_to_check = ['Symptom_1_itching', 'Symptom_2_skin_rash', 'Symptom_3_nodal_skin_eruptions']

for s in symptoms_to_check:
    if s in input_data.columns:
        input_data[s] = 1

# 4. Predict
prediction = model.predict(input_data)[0]
confidence = np.max(model.predict_proba(input_data)) * 100

print(f"--- Diagnosis Result ---")
print(f"Predicted Disease: {prediction}")
print(f"Confidence Level: {confidence:.2f}%")

--- Diagnosis Result ---
Predicted Disease: Heart attack
Confidence Level: 9.00%


In [9]:
import joblib
import pandas as pd
import numpy as np
from difflib import get_close_matches

# 1. Load the AI
model = joblib.load('disease_model.pkl')
known_symptoms = joblib.load('model_columns.pkl')

def get_prediction():
    print("\n--- 🏥 AI Health Assistant ---")
    print("Enter your symptoms one by one. Type 'done' when finished.")
    
    user_inputs = []
    while True:
        s = input("Enter Symptom: ").lower().strip()
        if s == 'done': break
        
        # Recognize Input: Find the closest match in our 600+ symptoms
        # We look for the symptom name inside the column names (e.g. 'Symptom_1_itching')
        match = get_close_matches(s, known_symptoms, n=1, cutoff=0.5)
        
        if match:
            print(f"✔️ Recognized as: {match[0]}")
            user_inputs.append(match[0])
        else:
            print("❌ Symptom not recognized. Please try a different word.")

    if not user_inputs:
        print("No symptoms entered. Exiting.")
        return

    # 2. Prepare the data for the Model
    # Create a row of all zeros
    input_row = pd.DataFrame(0, index=[0], columns=known_symptoms)
    
    # "Turn on" the symptoms the user provided
    for symptom in user_inputs:
        input_row[symptom] = 1

    # 3. Get Result
    prediction = model.predict(input_row)[0]
    probabilities = model.predict_proba(input_row)[0]
    confidence = np.max(probabilities) * 100

    # 4. Smart Output
    print("\n" + "="*30)
    print(f"DIAGNOSIS: {prediction}")
    print(f"CONFIDENCE: {confidence:.2f}%")
    
    if confidence < 30:
        print("⚠️ NOTE: Confidence is very low. Please provide more specific symptoms.")
    elif confidence > 80:
        print("✅ This matches the pattern for this disease very closely.")
    print("="*30)

# Run the checker
get_prediction()


--- 🏥 AI Health Assistant ---
Enter your symptoms one by one. Type 'done' when finished.


❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.
❌ Symptom not recognized. Please try a different word.


KeyboardInterrupt: Interrupted by user